# Consistency: GRPO on Countdown with a self-plausibility reward (3B)

Trains Llama model (after SFT) with GRPO on TinyZero 3- and 4-number puzzles from TinyZero. One flag, `ARM`, picks the reward:
`format_only` (format-only rewards, control), `mean` (format + the judge's per-token mean of solution+answer), `sum` (format + the judge's total log-prob of solution + answer),
`correctness` (format + the exact checker: the ceiling).
The judge is the SFT adapter trained with DPO on corrupted traces (empty, cut) which learns to predict which solutions are produced with intact traces.
Correctness and the Countdown monitors (made-up numbers, step arithmetic, reaching the target, answer vs solution) do not affect the rewards except the correctness arm.

Uses the following custom libraries to generate and verify Countdown solutions: `countdown.py`, `traces.py`, `rewards.py` from the repo's `countdown/`.

In [ ]:
# [config]
MODEL = "meta-llama/Llama-3.2-3B-Instruct"

SFT_RUN = "Llama-3.2-3B-Instruct-countdown3000-ep1-lr0.0001-seed0-tr2-judge3000-mix3x3000"

# The label-free judge from corrupted reasoning (Consistency_Corrupt_Judge, TRAIN_KINDS empty + cut), under SFT_RUN.
DPO_RUN = "judge_corrupt_empty-swap-cut_cut0.5_n4x1000_n3x1000_t0.9_seed0/dpo_empty-cut_b0.1_lr2e-05_ep1"
JUDGE_TAG = "-corrupt_" + DPO_RUN.split("/dpo_")[1].split("_b")[0]      # in calibration and run names

# Arm (reward), one per run:
#   "format_only"  the tag rewards: xmlcount + soft/strict format + expression (FORMAT_FUNCS in [rewards])
#   "mean"         format + judge: per-token mean log-prob of solution + "Answer: ..." + end of turn (as on GSM8K)
#   "sum"          format + judge: summed log-prob of the same text
#   "correctness"  format + the exact checker (2.0 if right): the verifiable-reward ceiling
# Every arm logs the judge scores, correctness and the Countdown monitors with weight 0.
ARM = "sum"
RUN_TAG = "34-fmtw1-istt2-il-const"     # the same tag for the format_only control (the judge weight is unused there)
EXPRESSION_REWARD = True        # also reward "the answer is a whole-number expression" (check_answer's form)
# Judge settings. PLAUSIBILITY_MAX_SOLUTION_TOKENS: longer solutions score as missing (Countdown solutions: 3 lines).
PLAUSIBILITY_MAX_SOLUTION_TOKENS = 128
# JUDGE_NEEDS_ANSWER: a completion without an <answer> scores as missing (else its steps alone are judged).
JUDGE_NEEDS_ANSWER = True
# MISSING_AT (arms mean / sum): the value of a completion the judge can't score (no solution, over the cap, no answer).
# "wrong_median": calibrating plausibility reward for a missing solution block to the median value of the scored wrong answers in calibration; "below_floor": just below the 1st percentile, worse than any wrong answer.
MISSING_AT = "wrong_median"

# Training puzzles: N_CANDIDATES TinyZero 4-number puzzles (seeded draw without the 200 test puzzles, the one-shot
# example and all SFT puzzles), minus a random held-out set ([held-out]), plus as many 3-number puzzles (TRAIN_MIX_3).
N_CANDIDATES = 4000
TRAIN_MIX_3 = 0.5               # share of 3-number puzzles in training (Gandhi et al. 2025: 0.5): a new seeded draw of
                                # 3-number ones (none in any eval set)

# INTERLEAVE_34: every step trains on half four-number, then half three-number puzzles, in a fixed order
# ([train]: four-number puzzles shuffled with seed SEED, three-number with SEED + 1; saved to LOG_DIR/train_order.csv;
# shuffle_dataset off).
INTERLEAVE_34 = True

# Training parameters:
MAX_COMPLETION_LENGTH = 1024 
SMOKE_RUN = False               # True: pipeline test (~10-15 min): 10 steps, 64 candidates, 16 calibration puzzles,
                                 # held-out curve every 5 steps, 50 test puzzles; own cache files (the real run rebuilds)
MAX_STEPS = 150
PASS_AT_K = 16                   # sampled pass@k (at PASS_TEMPERATURE) on PASS_N held-out puzzles (half from each
PASS_N = 100                     # curve stratum: heldout + unscored): before training (cached), at PASS_EVAL_STEPS, and
PASS_EVAL_STEPS = (50, 100)      # after training; 0 = skip. ~PASS_N x PASS_AT_K completions each time (diversity check)
RICH_EVAL_STEPS = (50,)          # also the full "after" evals at these steps ([early stop]): test set, held-out strata, mixed
                                 # 3/4-number test (LOG_DIR/eval_*_step{N}*); () = only at the end
PASS_TEMPERATURE = 0.9           # fixed across runs, whatever TEMPERATURE trains at (the cached "before" is at 0.9)
LEARNING_RATE = 2e-5
LR_SCHEDULE = "constant_with_warmup"
TEMPERATURE = 0.9                # sampling temperature for training groups and calibration
KL_BETA = 0.02                   # KL penalty to the starting policy (a frozen copy, adapter "ref"); 0 = off
NUM_GENERATIONS = 16            # completions per puzzle (a group)
COMPLETIONS_PER_STEP = 128       # NUM_GENERATIONS x puzzles per step
WARMUP_STEPS = 20
SCALE_REWARDS = False           # divide advantages by the group's reward SD (GRPO's default)
LOSS_TYPE, MAX_GRAD_NORM, EPSILON = "dapo", 0.2, 0.5
ADAM_BETA2, WEIGHT_DECAY = 0.99, 0.0
# vLLM importance sampling (TRL): "token_truncate" = per-token train/vLLM probability ratio capped at VLLM_IS_CLIP
VLLM_IS_MODE, VLLM_IS_CLIP = "token_truncate", 2.0
# Judge weight (arms mean / sum). JUDGE_WEIGHT_ANCHOR "format": JUDGE_WEIGHT_SCALE x within-group SD of the summed format
# rewards (all completions) / within-group SD of the judge value (scored completions); uses the calibration's saved samples (the shared calibration file
# is not changed). None = the file's coef (correctness spread x 1)
JUDGE_WEIGHT_ANCHOR, JUDGE_WEIGHT_SCALE = "format", 1.0
VLLM_GPU_MEMORY_UTILIZATION = 0.35
MICRO_BATCH = 16

# The test set and the SFT puzzles, exactly as Consistency_Countdown draws them.
N_TEST, N_SFT, N_SFT_VAL, SEED = 200, 4000, 100, 0

if SMOKE_RUN:
    N_CANDIDATES = 64

DRIVE = "/content/drive/MyDrive/Consistency"
CODE_DIR = f"{DRIVE}/code/countdown"
ADAPTER_PATH = f"{DRIVE}/countdown/sft/{SFT_RUN}/adapter"
JUDGE_ADAPTER_PATH = f"{DRIVE}/countdown/sft/{SFT_RUN}/{DPO_RUN}/adapter"
GRPO_DIR = f"{DRIVE}/countdown/grpo/{SFT_RUN}-judge_dpo-start_sft"   # split, calibration, "before" evals and runs
CALIBRATION_PATH = (f"{GRPO_DIR}/calibration_cap{PLAUSIBILITY_MAX_SOLUTION_TOKENS}"   # per puzzle mix: the spreads differ
                    + ("_needans" if JUDGE_NEEDS_ANSWER else "") + "_nopool"
                    + (f"_mix3_{TRAIN_MIX_3:g}" if TRAIN_MIX_3 else "") + JUDGE_TAG.replace("-", "_")
                    + ("_smoke" if SMOKE_RUN else "") + ".json")
JUDGE_ARMS = ("mean", "sum")     # the arms that train on the judge
RUN_NAME = (f"grpo-{ARM}" + ("-missmed" if ARM in JUDGE_ARMS and MISSING_AT == "wrong_median" else "")
            + (f"-mix3_{TRAIN_MIX_3:g}" if TRAIN_MIX_3 else "") + ("-expr" if EXPRESSION_REWARD else "")
            + (f"-kl{KL_BETA:g}" if KL_BETA else "") + (f"-t{TEMPERATURE:g}" if TEMPERATURE != 0.9 else "")
            + (JUDGE_TAG if ARM in JUDGE_ARMS else "") + ("-smoke" if SMOKE_RUN else "") + f"-{RUN_TAG}")
LOG_DIR = f"{GRPO_DIR}/runs/{RUN_NAME}"

assert ARM in ("format_only", "correctness") + JUDGE_ARMS, ARM
assert COMPLETIONS_PER_STEP % NUM_GENERATIONS == 0 and COMPLETIONS_PER_STEP % MICRO_BATCH == 0, "batch sizes don't divide"
assert MISSING_AT in ("below_floor", "wrong_median"), MISSING_AT
assert JUDGE_WEIGHT_ANCHOR in (None, "format"), JUDGE_WEIGHT_ANCHOR
assert not INTERLEAVE_34 or (TRAIN_MIX_3 == 0.5 and (COMPLETIONS_PER_STEP // NUM_GENERATIONS) % 2 == 0), \
    "INTERLEAVE_34 needs TRAIN_MIX_3 0.5 and an even number of puzzles per step"
assert not TRAIN_MIX_3 or 0 < TRAIN_MIX_3 < 1, "TRAIN_MIX_3: 0 < share < 1"
print(f"MODEL={MODEL}\nadapter={ADAPTER_PATH}\njudge={JUDGE_ADAPTER_PATH}  policy starts at sft"
      f"\nARM={ARM}  RUN_TAG={RUN_TAG}  SMOKE_RUN={SMOKE_RUN}  lr={LEARNING_RATE:g} ({LR_SCHEDULE})  steps={MAX_STEPS}"
      f"\nLOG_DIR={LOG_DIR}")

In [ ]:
# [drive + HF] Mount Drive and log in to HF (Colab Secret HF_TOKEN; Llama is gated). Re-running is harmless.
import os
from google.colab import drive, userdata
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # read by transformers and vLLM; no huggingface_hub import before [install]
for f in ("countdown.py", "prompts.py", "traces.py", "rewards.py"):
    assert os.path.exists(f"{CODE_DIR}/{f}"), f"{CODE_DIR}/{f} missing (copy countdown/ from the repo)"
assert os.path.exists(f"{ADAPTER_PATH}/adapter_model.safetensors"), f"no adapter at {ADAPTER_PATH}"
assert os.path.exists(f"{JUDGE_ADAPTER_PATH}/adapter_model.safetensors"), \
    f"no DPO adapter at {JUDGE_ADAPTER_PATH} (Consistency_Countdown [judge DPO])"

In [ ]:
# [install] Same packages as Consistency_GRPO, once per runtime: skipped when vllm and trl are installed. No restart:
# nothing they replace (torch, transformers) is imported before this cell. torchao is removed either way: Colab ships
# 0.10, and peft refuses to import below 0.16 (nothing here uses it).
import importlib.util, subprocess, sys
if importlib.util.find_spec("vllm") and importlib.util.find_spec("trl"):
    print("[install] skipped: vllm and trl are installed")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "trl[vllm]", "peft"], check=True)
    print("[install] done")
if importlib.util.find_spec("torchao"):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("[install] removed torchao")

In [ ]:
# [vLLM setup]
import os, sys
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0" 
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Colab's stdout/stderr have no OS file descriptor; vLLM asks for one during startup.
for stream, fd in ((sys.stdout, 1), (sys.stderr, 2)):
    try:
        stream.fileno()
    except Exception:
        stream.fileno = lambda fd=fd: fd

!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import importlib.metadata as md
for p in ["torch", "transformers", "trl", "peft", "datasets", "vllm"]:
    try:
        print(f"{p:14s}", md.version(p))
    except md.PackageNotFoundError:
        print(f"{p:14s}", "not installed")

In [ ]:
# [common] Tokenizer, the Countdown code (countdown.py, traces.py, rewards.py from CODE_DIR), and the puzzles:
#   test         the N_TEST puzzles of Consistency_Countdown's discrimination test (same draw), for before/after evals
#   SFT puzzles  the adapter's training puzzles (same draw as Consistency_Countdown), excluded from everything here
#   RL draw      a third seeded draw without both: N_CANDIDATES candidates (training + the random held-out set) +
#                UNSCORED_N more (never trained on: a held-out stratum)
import os, sys, re, json, random, importlib
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = "<|finetune_right_pad_id|>"   # not <|eot_id|> (the EOS)
tokenizer.chat_template = re.sub(r'strftime_now\([^)]*\)', '"26 Jul 2024"', tokenizer.chat_template)  # pinned date

def chat_prompt_ids(tok, messages) -> list[int]:
    """Templated prompt as token ids, no extra BOS (vLLM would add a second one to text)."""
    return tok(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
               add_special_tokens=False)["input_ids"]

sys.path.insert(0, CODE_DIR)

import prompts, countdown, traces, rewards
for _m in (prompts, countdown, traces, rewards):   # prompts first: the others import from it
    importlib.reload(_m)
from countdown import check_answer, solve
from prompts import countdown_prompt, countdown_question, bare_question
from rewards import tag, completion_text, diagnose

UNSCORED_N = 100
_tz = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", split="train")
_as_puzzle = lambda r: {"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])}
_key = lambda p: (tuple(sorted(p["numbers"])), p["target"])

# test + one-shot example: Consistency_Countdown's [puzzles] draw
_draw = _tz.select(np.random.default_rng(SEED).choice(len(_tz), size=min(len(_tz), 20 * N_TEST), replace=False))
_four = [_as_puzzle(r) for r in _draw if len(r["nums"]) == 4]
EXAMPLE, test_puzzles = _four[0], _four[1:N_TEST + 1]
_seen = {_key(p) for p in test_puzzles} | {_key(EXAMPLE)}
# the SFT puzzles: Consistency_Countdown's second draw
_sft = []
for r in _tz.select(np.random.default_rng(SEED + 1).choice(len(_tz), size=min(len(_tz), 6 * (N_SFT + N_SFT_VAL)),
                                                           replace=False)):
    p = _as_puzzle(r)
    if len(p["numbers"]) == 4 and _key(p) not in _seen:
        _seen.add(_key(p))
        _sft.append(p)
_sft = _sft[:N_SFT + N_SFT_VAL]
_seen = {_key(p) for p in test_puzzles} | {_key(EXAMPLE)} | {_key(p) for p in _sft}
# RL draw: the candidates, then the unscored held-out puzzles
_rl = []
for r in _tz.select(np.random.default_rng(SEED + 2).choice(len(_tz), size=min(len(_tz), 4 * (N_CANDIDATES + UNSCORED_N)),
                                                           replace=False)):
    p = _as_puzzle(r)
    if len(p["numbers"]) == 4 and _key(p) not in _seen:
        _seen.add(_key(p))
        _rl.append(p)
assert len(_rl) >= N_CANDIDATES + UNSCORED_N, f"only {len(_rl)} RL puzzles in the draw"

def puzzle_dataset(puzzles, first_id=0):
    """Rows for TRL: the chat prompt plus numbers / target (passed to the reward functions) and a puzzle id."""
    return Dataset.from_list([{"prompt": countdown_prompt(p["numbers"], p["target"]), "numbers": p["numbers"],
                               "target": p["target"], "pid": first_id + i} for i, p in enumerate(puzzles)])

candidates = puzzle_dataset(_rl[:N_CANDIDATES])                          # pid 0 .. N_CANDIDATES - 1
unscored = puzzle_dataset(_rl[N_CANDIDATES:N_CANDIDATES + UNSCORED_N], first_id=N_CANDIDATES)
test_set = puzzle_dataset(test_puzzles, first_id=10**6)                      # pids kept apart from the RL draw
# 3-number test puzzles from the same draw (none are in SFT or RL, which are 4-number only): with test_set, the 50/50
# 3- and 4-number mix Gandhi et al. 2025 train and evaluate on ([mixed test])
test3_set = puzzle_dataset([p for p in (_as_puzzle(r) for r in _draw if len(r["nums"]) == 3)
                            if _key(p) not in _seen][:N_TEST], first_id=2 * 10**6)
# 5-number test puzzles: generated (TinyZero has 3 and 4 numbers only; none in any SFT, judge or GRPO data), the same
# draw as Consistency_Countdown's [puzzles] (seed SEED + 5, solvable by construction, target 10-100): its 5-number judge
# eval used these first N_TEST5 ([5-number test])
N_TEST5 = 100
_r5 = random.Random(SEED + 5)
test5_set = puzzle_dataset([{"numbers": g["numbers"], "target": g["target"]}
                            for g in (countdown.generate(_r5, n_numbers=5, count_solutions=False) for _ in range(N_TEST5))],
                           first_id=3 * 10**6)
# TRAIN_MIX_3: 3-number training puzzles, a fourth seeded draw without every puzzle above and every 3-number puzzle of
# the test draw (test3_set and the ones after it); as many as make TRAIN_MIX_3 of the training set ([held-out] adds them)
train3 = None
if TRAIN_MIX_3:
    _n3 = round(TRAIN_MIX_3 / (1 - TRAIN_MIX_3) * (N_CANDIDATES - min(100, N_CANDIDATES // 5)))
    _no3 = (_seen | {_key(_as_puzzle(r)) for r in _draw}       # not the test draw, nor any puzzle of the SFT draw (the
            | {_key(_as_puzzle(r)) for r in _tz.select(np.random.default_rng(SEED + 1).choice(   # judge's 3-number
                len(_tz), size=min(len(_tz), 6 * (N_SFT + N_SFT_VAL)), replace=False))})       # DPO pairs come from it)
    _p3 = []
    for r in _tz.select(np.random.default_rng(SEED + 3).choice(len(_tz), size=min(len(_tz), 4 * _n3), replace=False)):
        p = _as_puzzle(r)
        if len(p["numbers"]) == 3 and _key(p) not in _no3:
            _no3.add(_key(p))
            _p3.append(p)
    assert len(_p3) >= _n3, f"only {len(_p3)} 3-number puzzles in the draw"
    train3 = puzzle_dataset(_p3[:_n3], first_id=5 * 10**6)
print(f"test {len(test_set)} (as in Consistency_Countdown) + {len(test3_set)} 3-number + {len(test5_set)} 5-number, SFT puzzles excluded {len(_sft)}, "
      f"candidates {len(candidates)}, unscored held-out {len(unscored)}")
print("example prompt:", candidates[0]["prompt"][0]["content"][:160], "...")

In [ ]:
# [model] The policy is a LoRA on the base model (adapter "default", trainable; GRPO continues the SFT adapter). The judge
# is the base model + the DPO adapter (DPO_RUN), loaded as a second, frozen adapter "judge"; judging switches to it under
# no_grad and back (judge_weights in [rewards]). No merging: both are single adapters on the bf16 base; with KL_BETA a
# third, frozen adapter "ref" (a copy of the starting SFT policy) is the KL reference.
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.bfloat16, device_map="auto")
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
_policy_init = ADAPTER_PATH
model = PeftModel.from_pretrained(model, _policy_init, is_trainable=True)          # adapter "default"
_cfg = model.peft_config["default"]
print(f"policy adapter (default) from {_policy_init}: r={_cfg.r} alpha={_cfg.lora_alpha} targets={sorted(_cfg.target_modules)}")
model.load_adapter(JUDGE_ADAPTER_PATH, adapter_name="judge", is_trainable=False)
model.set_adapter("default")                     # active = the policy; PEFT marks only "default" trainable
print(f"judge adapter (frozen) from {JUDGE_ADAPTER_PATH}")
if KL_BETA:   # TRL uses an adapter named "ref" as the KL reference: a frozen copy of the starting policy
    print(f"KL reference: TRL copies the starting policy adapter to 'ref' in [build], beta {KL_BETA:g}")


model.print_trainable_parameters()
assert all("lora_" in n for n, p in model.named_parameters() if p.requires_grad), "non-LoRA parameters are trainable"
assert any(p.requires_grad for n, p in model.named_parameters() if "lora_" in n), "adapter loaded frozen"
assert not any(p.requires_grad for n, p in model.named_parameters() if ".judge." in n or ".ref." in n), \
    "judge or ref adapter is trainable"

In [ ]:
# [rewards] Format rewards (xmlcount, soft and strict format), the Countdown rewards
# and monitors (rewards.py), the judge, and the step monitor (printed every step, completions dumped every 20 steps).
# The judge (the base + the frozen DPO adapter "judge") scores the <solution> text + "\nAnswer: <answer>" + end of turn, after a neutral system prompt + the bare puzzle, as
# the GSM8K reward does (solutions over PLAUSIBILITY_MAX_SOLUTION_TOKENS, and with JUDGE_NEEDS_ANSWER completions without
# an answer, score as missing). Two raw scores from one forward pass:
#   mean   per-token mean log-prob
#   sum    summed log-prob
# The trained plausibility value, per ARM, from [reward calibration] (CAL; floors clip outliers; missing = the MISSING_AT
# level: the median wrong answer, or just below the floor):
#   mean        max(mean, floor)
#   sum         max(sum, floor)

import math, inspect, contextlib
from trl.trainer.utils import selective_log_softmax
import rewards as R

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>") == 1:
        count += 0.125
    if text.count("</reasoning>") == 1:
        count += 0.125
    if text.count("<solution>") == 1:
        count += 0.005
    if text.count("</solution>") == 1:
        count += 0.005
    if text.count("<answer>") == 1:
        count += 0.125
    if text.count("</answer>") == 1:
        count += 0.125
    if "</answer>" in text:
        # Everything after the FIRST </answer> is penalized, so repeating the tag can't switch the penalty off.
        count -= (len(text.split("</answer>", 1)[1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    return [count_xml(completion_text(c)) for c in completions]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>[\s\S]*?</reasoning>\s*<solution>[\s\S]*?</solution>\s*<answer>[\s\S]*?</answer>"
    return [0.5 if re.match(pattern, completion_text(c)) else 0.0 for c in completions]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>\s*.*?\s*</reasoning>\s*<solution>\s*.*?\s*</solution>\s*<answer>\s*.*?\s*</answer>"
    return [0.5 if re.search(pattern, completion_text(c), re.DOTALL) else 0.0 for c in completions]

FORMAT_FUNCS = ([xmlcount_reward_func, soft_format_reward_func, strict_format_reward_func]
                + ([R.expression_reward_func] if EXPRESSION_REWARD else []))   # 0.5 if the answer is a bare expression

# ---- judge
NEUTRAL_SYSTEM_PROMPT = "Please solve the following problem step-by-step, making sure that each step is well-justified."

@contextlib.contextmanager
def judge_weights(model):
    """The judge's weights for the duration (the frozen "judge" adapter); then back to the policy adapter, which alone
    is trainable."""
    model.set_adapter("judge")
    try:
        yield
    finally:
        model.set_adapter("default")
        for n, p in model.named_parameters():        # set_adapter makes the active adapter trainable: only the policy is
            if "lora_" in n and ".default." not in n:
                p.requires_grad_(False)

def judge_raw(owner, prompts, completions, chunk_size=8):
    """(mean, sum) per completion as float arrays, NaN where there is nothing to score (no solution, over the cap, or
    no answer with JUDGE_NEEDS_ANSWER). Scored under owner.model with the judge's weights (judge_weights)."""
    tok, model = owner.processing_class, owner.model
    fwd = getattr(model, "_original_forward", model.forward)   # skip accelerate's autocast + fp32 logits copy
    means, sums = (np.full(len(completions), np.nan) for _ in range(2))
    items = []
    for i, (prompt, c) in enumerate(zip(prompts, completions)):
        text = completion_text(c)
        sol, ans = tag(text, "solution"), tag(text, "answer")
        if not sol or len(tok(sol, add_special_tokens=False)["input_ids"]) > PLAUSIBILITY_MAX_SOLUTION_TOKENS:
            continue
        if JUDGE_NEEDS_ANSWER and not ans:
            continue                                            # missing, like no solution
        target = sol + (f"\nAnswer: {ans}" if ans else "")
        enc = tok(target, add_special_tokens=False)
        p_ids = chat_prompt_ids(tok, [{"role": "system", "content": NEUTRAL_SYSTEM_PROMPT},
                                      {"role": "user", "content": bare_question(prompt)}])
        items.append((i, p_ids, enc["input_ids"] + [tok.eos_token_id]))
    with torch.no_grad(), judge_weights(model):
        for start in range(0, len(items), chunk_size):
            chunk = items[start:start + chunk_size]
            L = max(len(p) + len(t) for _, p, t in chunk)
            ids = torch.full((len(chunk), L), tok.pad_token_id, dtype=torch.long)
            attn = torch.zeros_like(ids)
            for j, (_, p, t) in enumerate(chunk):
                ids[j, :len(p) + len(t)] = torch.tensor(p + t)
                attn[j, :len(p) + len(t)] = 1
            ids, attn = ids.to(model.device), attn.to(model.device)
            per_pos = selective_log_softmax(fwd(input_ids=ids, attention_mask=attn).logits[:, :-1], ids[:, 1:]).float()
            for j, (i, p, t) in enumerate(chunk):
                lps = per_pos[j, len(p) - 1:len(p) + len(t) - 1].cpu().numpy()   # positions predicting t
                means[i], sums[i] = lps.mean(), lps.sum()
            del per_pos
    return means, sums

CAL = json.load(open(CALIBRATION_PATH)) if os.path.exists(CALIBRATION_PATH) else None

def plausibility_value(m, sm, arm):
    """The trained plausibility value from raw mean / sum (arrays); before calibration: raw mean, NaN -> None.
    Missing (NaN) gets the calibration's MISSING_AT level."""
    if CAL is None:
        return [None if math.isnan(x) else float(x) for x in m]
    miss = lambda c: c.get(f"missing_{MISSING_AT}", c.get("missing"))   # "missing": files from before MISSING_AT
    if arm == "sum":
        c = CAL["sum"]
        return [miss(c) if math.isnan(x) else float(max(x, c["floor"])) for x in sm]
    c = CAL["mean"]
    return [miss(c) if math.isnan(x) else float(max(x, c["floor"])) for x in m]

_LAST_JUDGE = {}   # this batch's raw scores, for the monitors below (matched by the identity of `completions`)

def make_plausibility(owner):
    arm = ARM if ARM in JUDGE_ARMS else "mean"
    def plausibility(prompts, completions, **kwargs):
        m, sm = judge_raw(owner, prompts, completions)
        _LAST_JUDGE.update(id=id(completions), mean=m, sum=sm)
        return plausibility_value(m, sm, arm)
    plausibility.__name__ = "plausibility"
    return plausibility

def plausibility_weight():
    """Training weight: the calibrated coefficient in the plausibility arms, 0 otherwise or before calibration."""
    return float(CAL[ARM]["coef"]) if (CAL is not None and ARM in JUDGE_ARMS and ARM in CAL) else 0.0

def _from_last(key, completions):
    v = _LAST_JUDGE.get(key) if _LAST_JUDGE.get("id") == id(completions) else None
    return [None] * len(completions) if v is None else [None if math.isnan(x) else float(x) for x in v]

def judge_mean(completions, **kwargs):
    return _from_last("mean", completions)

def judge_sum(completions, **kwargs):
    return _from_last("sum", completions)

def reasoning_tokens(completions, **kwargs):
    return [None if (r := tag(completion_text(c), "reasoning")) is None
            else float(len(tokenizer(r, add_special_tokens=False)["input_ids"])) for c in completions]

def solution_tokens(completions, **kwargs):
    return [None if (s := tag(completion_text(c), "solution")) is None
            else float(len(tokenizer(s, add_special_tokens=False)["input_ids"])) for c in completions]

# ---- the step monitor: prints this step's means and the judge's within-puzzle win rate, dumps completions every 20 steps
def within_puzzle_win_rate(keys, scores, correct):
    """Over (correct, wrong) pairs of the same puzzle (NaN scores left out): share where the correct one scores
    higher, ties 1/2. Returns (rate, pairs)."""
    wins = pairs = 0
    keys = np.array(keys)
    for k in set(keys.tolist()):
        s, c = scores[keys == k], correct[keys == k]
        cs, ws = s[c & ~np.isnan(s)], s[~c & ~np.isnan(s)]
        d = cs[:, None] - ws[None, :]
        wins += (d > 0).sum() + 0.5 * (d == 0).sum()
        pairs += d.size
    return (wins / pairs if pairs else float("nan")), int(pairs)

def make_step_monitor(owner):
    def step_monitor(prompts, completions, numbers, target, pid, **kwargs):
        correct = np.array(R.correctness_reward_func(prompts, completions, numbers, target)) > 0
        fmt = sum(np.array(f(completions=completions, numbers=numbers, target=target), dtype=float) for f in FORMAT_FUNCS)
        m = np.array(judge_mean(completions), dtype=float)
        cc = np.array([np.nan if x is None else x for x in R.caught_claims(completions)], dtype=float)
        st, gu = np.mean(R.stopped(completions)), np.mean(R.gave_up(completions))
        avg = lambda x: f"{np.nanmean(x):.2f}" if np.isfinite(x).any() else "nan"
        line = (f"sampled T {TEMPERATURE:g}  correct {correct.mean():.3f}  format {fmt.mean():.3f}"
                + (f"  stopped {st:.3f}" if st else "") + (f"  gave up {gu:.3f}" if gu else "")
                + f"  caught claims {avg(cc)}")
        rate, pairs = within_puzzle_win_rate(pid, m, correct)
        line += f"  | judge mean: correct {avg(m[correct])}  wrong {avg(m[~correct])}  win {rate:.2f} ({pairs})"
        if pairs and kwargs.get("log_metric"):
            kwargs["log_metric"]("judge_mean_win_rate", float(rate))
        step = owner.state.global_step
        print(f"[step {step}] {line}")
        if step % 20 == 0:
            os.makedirs(LOG_DIR, exist_ok=True)
            with open(f"{LOG_DIR}/step_{step:05d}.txt", "w", encoding="utf-8") as f:
                f.write(f"=== Step {step} ===\n\n")
                for i, (p, c) in enumerate(zip(prompts, completions)):
                    f.write(f"### Sample {i + 1}\n[Scores] puzzle={pid[i]} correct={bool(correct[i])} "
                            f"judge_mean={m[i]:.4f}\n[Prompt]\n{p[-1]['content'].strip()}\n\n"
                            f"[Response]\n{completion_text(c).strip()}\n\n" + "=" * 80 + "\n\n")
        return [0.0] * len(completions)
    step_monitor.__name__ = "step_monitor"
    return step_monitor

# (function, weight); "plausibility" and "step_monitor" are bound to the trainer in [build]. Order matters: the judge
# monitors read the scores the plausibility reward just computed.
REWARD_FUNCS_TRAIN = ([(f, 1.0) for f in FORMAT_FUNCS]
                      + [("plausibility", plausibility_weight()),
                         (R.correctness_reward_func, 1.0 if ARM == "correctness" else 0.0),
                         (judge_mean, 0.0), (judge_sum, 0.0)]
                      + [(f, 0.0) for f in R.MONITORS + [reasoning_tokens, solution_tokens]]
                      + [("step_monitor", 0.0)])
print("reward functions (weight):", [(f if isinstance(f, str) else f.__name__, w) for f, w in REWARD_FUNCS_TRAIN])
print("calibration:", CALIBRATION_PATH if CAL else "none yet (run [reward calibration]; plausibility weight 0 until then)")

In [ ]:
# [vLLM sync fix]
# TRL syncs LoRA to vLLM by merging dW into the training model's bf16 W in place, streaming it, then unmerging.
# Each bf16 add/subtract rounds, so the "frozen" W drifts a little at every sync (seen: 0.16% of weights after 10 steps).
# This version never touches the training model: for each LoRA layer it builds bf16(W + dW) in a temporary tensor
# (dW and the sum in fp32, one rounding), streams it to vLLM, and frees it. Peak extra memory: one layer in fp32.
import torch
from accelerate.utils import is_peft_model
from peft.tuners.lora import LoraLayer
from trl.generation.vllm_generation import VLLMGeneration

if not hasattr(VLLMGeneration, "_trl_iter_named_params"):
    VLLMGeneration._trl_iter_named_params = VLLMGeneration._iter_named_params  # keep TRL's original

def _iter_named_params_out_of_place(self):
    model = self.model
    if not is_peft_model(model) or self.accelerator.num_processes > 1:
        yield from self._trl_iter_named_params()  # not the single-GPU LoRA case this fix is for
        return
    lora_layers = {name: m for name, m in model.named_modules() if isinstance(m, LoraLayer)}
    n_merged = 0
    for name, param in model.named_parameters():
        vllm_name = name.removeprefix("base_model.model.").replace(".base_layer", "")
        if model.prefix in vllm_name or "original_module" in vllm_name:
            continue  # LoRA A/B themselves don't exist in vLLM
        vllm_name = self._fix_param_name_to_vllm(vllm_name, extra_prefixes=["modules_to_save.default."])
        layer = lora_layers.get(name.rsplit(".base_layer.", 1)[0]) if name.endswith(".base_layer.weight") else None
        if layer is None:
            yield vllm_name, param.data
            continue
        assert not layer.merged, f"{name}: adapter is merged in the training model"
        with torch.no_grad():
            merged = param.data.float()
            for adapter in layer.active_adapters:
                if adapter in layer.lora_A:
                    merged += layer.get_delta_weight(adapter).float()
            merged = merged.to(param.dtype)
        n_merged += 1
        yield vllm_name, merged
        del merged
    if not getattr(self, "_out_of_place_reported", False):
        print(f"[vLLM sync fix] streamed {n_merged} LoRA layers as bf16(W + dW); training weights untouched")
        self._out_of_place_reported = True

VLLMGeneration._iter_named_params = _iter_named_params_out_of_place
print("VLLMGeneration._iter_named_params patched")

In [ ]:
# [eval] Greedy eval of any puzzle set on the trainer's vLLM engine (sync the weights first: before training the
# engine holds the base weights). Per completion: correct (check_answer), what is wrong (diagnose), format,
# truncation, the Countdown monitors. Used for the test set ([build] before, [train] after), the held-out strata and
# the held-out curve.
from vllm import SamplingParams

_FORMAT_RE = re.compile(r"<reasoning>.*</reasoning>\s*<solution>.*</solution>\s*<answer>.*</answer>", re.DOTALL)

def eval_puzzles(trainer, ds, tag_, out_dir=None, max_tokens=None, quiet=False):
    """Greedy completions for the puzzle Dataset ds; returns (summary dict, per-puzzle DataFrame)."""
    ids = [chat_prompt_ids(tokenizer, p) for p in ds["prompt"]]
    outs = trainer.vllm_generation.llm.generate(
        [{"prompt_token_ids": x} for x in ids],
        SamplingParams(temperature=0.0, max_tokens=max_tokens or MAX_COMPLETION_LENGTH), use_tqdm=not quiet)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(outs, ids)), "vLLM returned outputs for other prompts"
    texts = [o.outputs[0].text for o in outs]
    wrapped = [[{"role": "assistant", "content": t}] for t in texts]
    df = pd.DataFrame({"pid": ds["pid"], "numbers": [json.dumps(n) for n in ds["numbers"]], "target": ds["target"],
                       "completion": texts, "tokens": [len(o.outputs[0].token_ids) for o in outs],
                       "truncated": [o.outputs[0].finish_reason == "length" for o in outs],
                       "format_ok": [bool(_FORMAT_RE.search(t)) for t in texts]})
    df["answer"] = [tag(t, "answer") for t in texts]
    df["correct"] = np.array(R.correctness_reward_func(None, wrapped, ds["numbers"], ds["target"])) > 0
    df["kind"] = [diagnose(a, n, t, tr) for a, n, t, tr in zip(df.answer, ds["numbers"], ds["target"], df.truncated)]
    for f in R.MONITORS:
        df[f.__name__] = np.array(f(completions=wrapped, numbers=ds["numbers"], target=ds["target"]), dtype=float)
    summary = {"tag": tag_, "n": len(df), "accuracy": df.correct.mean(),
               "accuracy_finished": df.loc[~df.truncated, "correct"].mean(), "truncated": df.truncated.mean(),
               "format_ok": df.format_ok.mean(), "tokens": df.tokens.mean(),
               **{f.__name__: df[f.__name__].mean() for f in R.MONITORS},
               **{f"kind/{k}": v for k, v in df.kind.value_counts(normalize=True).items()}}
    if not quiet:
        print(f"[eval:{tag_}] greedy, n={len(df)}  accuracy {summary['accuracy']:.3f} (finished-only "
              f"{summary['accuracy_finished']:.3f})  truncated {summary['truncated']:.3f}  format {summary['format_ok']:.3f}  "
              f"tokens {summary['tokens']:.0f}\n  " + ", ".join(f"{k[5:]} {v:.2f}" for k, v in summary.items()
                                                                 if k.startswith("kind/") and k != "kind/correct"))
    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)
        json.dump({k: (float(v) if isinstance(v, (np.floating, np.integer, bool)) else v) for k, v in summary.items()},
                  open(f"{out_dir}/eval_{tag_}.json", "w"), indent=2)
        df.to_csv(f"{out_dir}/eval_{tag_}_samples.csv", index=False)
    return summary, df

def pass_at_k(trainer, ds, tag_, k, out_dir=None, strata=None, temperature=None):
    """k sampled completions per puzzle at temperature (default PASS_TEMPERATURE): pass@1 (mean accuracy), pass@k (any correct),
    distinct correct answers per puzzle; overall and per stratum (strata: pid -> name). Saves eval_{tag_}_pass{k}.*"""
    ids = [chat_prompt_ids(tokenizer, p) for p in ds["prompt"]]
    outs = trainer.vllm_generation.llm.generate(
        [{"prompt_token_ids": x} for x in ids],
        SamplingParams(n=k, temperature=temperature or PASS_TEMPERATURE, max_tokens=MAX_COMPLETION_LENGTH, seed=0), use_tqdm=True)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(outs, ids)), "vLLM returned outputs for other prompts"
    rows = []
    for o, pid, nums, tgt in zip(outs, ds["pid"], ds["numbers"], ds["target"]):
        texts = [c.text for c in o.outputs]
        ok = np.array(R.correctness_reward_func(None, [[{"role": "assistant", "content": t}] for t in texts],
                                                [nums] * k, [tgt] * k)) > 0
        right = {re.sub(r"\s+", "", tag(t, "answer") or "") for t, c in zip(texts, ok) if c}
        rows.append({"pid": pid, "stratum": (strata or {}).get(pid, "other"), "correct": int(ok.sum()),
                     "pass1": ok.mean(), f"pass{k}": float(ok.any()), "distinct_correct": len(right),
                     "truncated": float(np.mean([c.finish_reason == "length" for c in o.outputs]))})
    df = pd.DataFrame(rows)
    cols = ["pass1", f"pass{k}", "distinct_correct", "truncated"]
    summary = pd.concat([df[cols].mean().rename("total").to_frame().T, df.groupby("stratum")[cols].mean()])
    summary["n"] = [len(df)] + df.groupby("stratum").size().tolist()
    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)
        df.to_csv(f"{out_dir}/eval_{tag_}_pass{k}_samples.csv", index=False)
        summary.to_csv(f"{out_dir}/eval_{tag_}_pass{k}.csv")
    return summary, df

def eval_mixed(trainer, tag_, out_dir=None, test4=None):
    """Greedy accuracy on Gandhi et al. 2025's mix: test3_set (3-number) and the 4-number test set, mixed = their mean.
    test4: an existing (summary, df) greedy eval of the SAME weights on the full test_set; reused (saved again as
    test4_{tag_}) instead of a second eval of the same puzzles. Returns (mixed summary, 3-number summary, 4-number summary)."""
    s3, _ = eval_puzzles(trainer, test3_set, f"test3_{tag_}", out_dir=out_dir, quiet=True)
    if test4 is not None and len(test4[1]) == len(test_set):
        s4, d4 = {**test4[0], "tag": f"test4_{tag_}"}, test4[1]
        if out_dir is not None:
            json.dump({k: (float(v) if isinstance(v, (np.floating, np.integer, bool)) else v) for k, v in s4.items()},
                      open(f"{out_dir}/eval_test4_{tag_}.json", "w"), indent=2)
            d4.to_csv(f"{out_dir}/eval_test4_{tag_}_samples.csv", index=False)
    else:
        s4, _ = eval_puzzles(trainer, test_set, f"test4_{tag_}", out_dir=out_dir, quiet=True)
    mixed = {"3-number": s3["accuracy"], "4-number": s4["accuracy"], "mixed": (s3["accuracy"] + s4["accuracy"]) / 2,
             "truncated 3": s3["truncated"], "truncated 4": s4["truncated"]}
    return mixed, s3, s4

# ---- reporting: one table per eval point; the weights it measures are named in the header ----
CURRENT_WEIGHTS = globals().get("CURRENT_WEIGHTS", "starting adapter")   # what the session's policy holds ([train] sets it)

def curve_of(out_dir, tag_):
    """Greedy summary of the held-out curve puzzles (CURVE_STRATA, [held-out]) from an eval_heldout(..., tag_, out_dir)."""
    df = pd.concat([pd.read_csv(f"{out_dir}/eval_heldout_{s}_{tag_}_samples.csv") for s in CURVE_STRATA])
    fin = ~df.truncated.astype(bool)
    return {"accuracy": float(df.correct.mean()), "accuracy_finished": float(df.loc[fin, "correct"].mean()),
            "truncated": float(df.truncated.mean()), "format_ok": float(df.format_ok.mean()),
            "tokens": float(df.tokens.mean()), "n": len(df)}

def _mixed_before_dir():
    return f"{GRPO_DIR}/eval_mixed_before_n{len(test3_set)}_max{MAX_COMPLETION_LENGTH}"

def mixed_before_cached():
    """The starting adapter's 3- and 4-number test evals (greedy) if cached, else None."""
    d = _mixed_before_dir()
    if not os.path.exists(f"{d}/summary.json"):
        return None
    return {k: json.load(open(f"{d}/eval_{k}_before.json")) for k in ("test3", "test4")}

def mixed_before_evals(trainer):
    """As mixed_before_cached; if not cached, evals the starting adapter (loaded for the eval, then the session's
    weights are restored) and caches it in GRPO_DIR."""
    if mixed_before_cached() is None:
        from peft import get_peft_model_state_dict, set_peft_model_state_dict
        from safetensors.torch import load_file
        d = _mixed_before_dir()
        _now = {k: v.detach().clone() for k, v in get_peft_model_state_dict(trainer.model, adapter_name="default").items()}
        set_peft_model_state_dict(trainer.model, load_file(f"{_policy_init}/adapter_model.safetensors"), adapter_name="default")
        trainer.vllm_generation.sync_weights()
        try:
            mixed, _, _ = eval_mixed(trainer, "before", d)
        finally:
            set_peft_model_state_dict(trainer.model, _now, adapter_name="default")
            trainer.vllm_generation.sync_weights()
        json.dump(mixed, open(f"{d}/summary.json", "w"), indent=2)
    return mixed_before_cached()

_SETS = (("test4", "4-number test"), ("test3", "3-number test"), ("curve", "held-out curve puzzles"))

def report(title, weights, cols, out=None):
    """Prints ONE table: rows = (set, decoding, puzzles x samples, metric), columns = cols' keys (e.g. "start", "step 50").
    cols: {column: results}; results: dict with any of "test4", "test3", "curve" (greedy eval summaries) and "pass"
    (pass_at_k summary, sampled). Saved to out (csv) if given."""
    rows = {}
    for key, name in _SETS:
        for c, res in cols.items():
            s = (res or {}).get(key)
            if s is None:
                continue
            for m in ("accuracy", "truncated"):
                rows.setdefault((name, "greedy", f"{int(s.get('n', 0))} x 1", m), {})[c] = float(s[m])
    for c, res in cols.items():
        p = (res or {}).get("pass")
        if p is None:
            continue
        t = p.loc["total"]
        for m, lab in (("pass1", "pass@1"), (f"pass{PASS_AT_K}", f"pass@{PASS_AT_K}"),
                       ("distinct_correct", "distinct correct answers"), ("truncated", "truncated")):
            rows.setdefault((f"pass@{PASS_AT_K} puzzles (in curve set)", f"T {PASS_TEMPERATURE:g}",
                             f"{int(t['n'])} x {PASS_AT_K}", lab), {})[c] = float(t[m])
    if not rows:
        print(f"\n[report] {title} | weights: {weights}: nothing to report")
        return None
    df = pd.DataFrame.from_dict(rows, orient="index").reindex(columns=list(cols))
    df.index = pd.MultiIndex.from_tuples(df.index, names=["set", "decoding", "puzzles x samples", "metric"])
    print(f"\n[report] {title} | weights: {weights}")
    with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 200):
        print(df.to_string(na_rep="-"))
    if out is not None:
        df.to_csv(out)
    return df

def curve_line(curve):
    """The held-out curve as one line: step -> greedy accuracy."""
    n = len(CURVE_SET) if "CURVE_SET" in globals() else "?"
    return (f"[held-out curve] held-out curve puzzles, greedy, {n} x 1, accuracy by step: "
            + "  ".join(f"{int(r['step'])}: {r['accuracy']:.3f}" for r in curve))

In [ ]:
# [build] GRPO config + trainer (rewards from [rewards]) + the "before" test eval (cached in GRPO_DIR). Then:
# [held-out] -> [reward calibration] -> [early stop] -> [train]. Settings as Consistency_GRPO's 8B run b.
import csv, shutil, tempfile
from datetime import datetime
from transformers import TrainerCallback
from trl import GRPOTrainer, GRPOConfig

class CSVLoggerCallback(TrainerCallback):
    """Every log row to a CSV; the header grows when new keys appear (rewritten in place)."""
    def __init__(self, path):
        self.path, self.fields = path, None
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if os.path.exists(path) and os.path.getsize(path):
            with open(path, newline="") as f:
                self.fields = next(csv.reader(f))
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        logs = {"step": state.global_step, **logs}
        if self.fields is None:
            self.fields = list(logs)
            with open(self.path, "w", newline="") as f:
                csv.DictWriter(f, fieldnames=self.fields).writeheader()
        new = [k for k in logs if k not in self.fields]
        if new:
            self.fields += sorted(new)
            rows = list(csv.DictReader(open(self.path, newline="")))
            fd, tmp = tempfile.mkstemp(dir=os.path.dirname(self.path))
            os.close(fd)
            with open(tmp, "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=self.fields)
                w.writeheader()
                w.writerows(rows)
            shutil.move(tmp, self.path)
        with open(self.path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=self.fields).writerow(logs)

training_args = GRPOConfig(
    use_vllm=True, vllm_mode="colocate", vllm_gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    vllm_max_model_length=MAX_COMPLETION_LENGTH + 512,     # prompt (~110 tokens) + completion
    gradient_checkpointing=True,
    loss_type=LOSS_TYPE, scale_rewards=SCALE_REWARDS, max_grad_norm=MAX_GRAD_NORM, epsilon=EPSILON, temperature=TEMPERATURE,
    beta=KL_BETA,
    vllm_importance_sampling_correction=True, vllm_importance_sampling_mode=VLLM_IS_MODE,
    vllm_importance_sampling_clip_max=VLLM_IS_CLIP,
    reward_weights=[w for _, w in REWARD_FUNCS_TRAIN],
    learning_rate=LEARNING_RATE, adam_beta1=0.9, adam_beta2=ADAM_BETA2, weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS, lr_scheduler_type=LR_SCHEDULE, optim="adamw_torch",
    shuffle_dataset=not INTERLEAVE_34,                     # INTERLEAVE_34: [train] sets the order
    logging_strategy="steps", logging_steps=5,
    save_strategy="steps", save_steps=25, save_total_limit=None,   # every checkpoint kept: [early stop] picks the best
    output_dir=LOG_DIR, report_to=["tensorboard"],
    bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
    per_device_train_batch_size=MICRO_BATCH, gradient_accumulation_steps=COMPLETIONS_PER_STEP // MICRO_BATCH,
    num_generations=NUM_GENERATIONS, max_completion_length=MAX_COMPLETION_LENGTH,
    max_steps=10 if SMOKE_RUN else MAX_STEPS, eval_strategy="no",
)

class CountdownGRPOTrainer(GRPOTrainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # the judge and the step monitor need the trainer (model, tokenizer, step): bind them now
        for i, f in enumerate(self.reward_funcs):
            if getattr(f, "__name__", "") == "plausibility":
                self.reward_funcs[i] = make_plausibility(self)
            elif getattr(f, "__name__", "") == "step_monitor":
                self.reward_funcs[i] = make_step_monitor(self)

    def log(self, logs, start_time=None):
        super().log(logs, start_time)
        if "reward" in logs and "train_runtime" not in logs:
            v = lambda k, d=float("nan"): d if logs.get(k) is None else float(logs[k])   # TRL logs None for all-NaN
            print(f"[{self.state.global_step}] loss={v('loss', 0.0):.4f}, reward={v('reward'):.4f}, "
                  f"grad_norm={v('grad_norm'):.3f}, entropy={v('entropy'):.3f}, "
                  f"correct={v('rewards/correctness_reward_func/mean') / 2:.3f}, "
                  f"reasoning={v('rewards/reasoning_tokens/mean'):.0f}")

def _placeholder(name):
    f = lambda completions, **kwargs: [0.0] * len(completions)
    f.__name__ = name
    return f

trainer = CountdownGRPOTrainer(
    model=model, processing_class=tokenizer, args=training_args, train_dataset=candidates,   # [held-out] sets it
    reward_funcs=[_placeholder(f) if isinstance(f, str) else f for f, _ in REWARD_FUNCS_TRAIN])
trainer.add_callback(CSVLoggerCallback(f"{LOG_DIR}/metrics.csv"))
print("reward weights:", {getattr(f, "__name__", "?"): float(w) for f, w in zip(trainer.reward_funcs, trainer.reward_weights)})

# The test set "before" (the starting adapter, greedy) doesn't depend on the arm: cached in GRPO_DIR, copied per run.
EVAL_MAX = 200 if not SMOKE_RUN else 50
_test = test_set.select(range(min(EVAL_MAX, len(test_set))))

trainer.vllm_generation.sync_weights()      # the engine starts with base weights

# Sampler check: vLLM (what generates) and the trainer (what computes the policy's log-probs) must be the same model.
# Greedy completions of a few test puzzles, each token's log-prob from both. Runs a-c: mean |difference| ~0.003 per
# token (bf16); the merged-judge version of 2026-10-16 had ~0.012 and an importance-sampling ratio of ~0.4.
SAMPLER_MAX_DIFF = 0.006
def sampler_check(n_prompts=8, max_tokens=128):
    ds = _test.select(range(min(n_prompts, len(_test))))
    ids = [chat_prompt_ids(tokenizer, p) for p in ds["prompt"]]
    outs = trainer.vllm_generation.llm.generate([{"prompt_token_ids": x} for x in ids],
                                                SamplingParams(temperature=0.0, max_tokens=max_tokens, logprobs=1),
                                                use_tqdm=False)
    diffs = []
    for x, o in zip(ids, outs):
        toks = list(o.outputs[0].token_ids)
        v = np.array([lp[t].logprob for lp, t in zip(o.outputs[0].logprobs, toks)])
        inp = torch.tensor([x + toks], device=trainer.model.device)
        with torch.no_grad():
            logits = trainer.model(input_ids=inp).logits[0, len(x) - 1:-1].float()
        hf = torch.log_softmax(logits, -1).gather(1, inp[0, len(x):, None])[:, 0].cpu().numpy()
        diffs.append(np.abs(hf - v))
    d = np.concatenate(diffs)
    return float(d.mean()), float(d.max()), len(d)
_sc_mean, _sc_max, _sc_n = sampler_check()
print(f"[sampler check] vLLM vs trainer log-probs over {_sc_n} tokens: mean |diff| {_sc_mean:.4f}, max {_sc_max:.3f} "
      f"(runs a-c ~0.003)")
assert _sc_mean < SAMPLER_MAX_DIFF, (f"vLLM and the trainer disagree ({_sc_mean:.4f} > {SAMPLER_MAX_DIFF}): the weights "
                                     f"vLLM generates with are not the policy the trainer updates")
_before = f"{GRPO_DIR}/eval_test_before_n{len(_test)}_max{MAX_COMPLETION_LENGTH}"
if os.path.exists(f"{_before}.json"):
    eval_before = json.load(open(f"{_before}.json"))
    print(f"[eval:before] 4-number test, greedy, {int(eval_before['n'])} x 1, starting adapter (cached {_before}.json): "
          f"accuracy {eval_before['accuracy']:.3f}, truncated {eval_before['truncated']:.3f}")
else:
    eval_before, _ = eval_puzzles(trainer, _test, "test_before", out_dir=GRPO_DIR)
    for ext in (".json", "_samples.csv"):
        shutil.move(f"{GRPO_DIR}/eval_test_before{ext}", f"{_before}{ext}")
os.makedirs(LOG_DIR, exist_ok=True)
for ext in (".json", "_samples.csv"):
    shutil.copy(f"{_before}{ext}", f"{LOG_DIR}/eval_test_before{ext}")

In [ ]:
# [held-out] Eval frames that are never trained on (fixed, seed HELDOUT_SEED; saved in SPLIT_OUT, shared by all arms).
#   heldout       a random set of the candidates, removed from training here (same distribution as training)
#   unscored      the UNSCORED_N extra puzzles of the RL draw
# The test set (Consistency_Countdown's 200) stays the main before/after eval. Greedy, MAX_COMPLETION_LENGTH.
from datasets import concatenate_datasets
HELDOUT_N, HELDOUT_SEED = 100, 1
SPLIT_OUT = f"{GRPO_DIR}/grpo_split_n{N_CANDIDATES}_nopool_seed{HELDOUT_SEED}.json"

if os.path.exists(SPLIT_OUT):
    grpo_split = json.load(open(SPLIT_OUT))
    print(f"[held-out] split loaded: {SPLIT_OUT}")
else:
    r = np.random.default_rng(HELDOUT_SEED)
    pick = lambda ids, n: sorted(r.choice(sorted(ids), size=min(n, len(ids)), replace=False).tolist())
    all_ids = set(candidates["pid"])
    grpo_split = {"heldout": pick(all_ids, min(HELDOUT_N, len(all_ids) // 5)), "unscored": sorted(unscored["pid"])}
    grpo_split["train"] = sorted(all_ids - set(grpo_split["heldout"]))
    grpo_split["note"] = ("puzzle ids (pid) of the RL draw; training = train (all candidates, no pass-rate filter); "
                          "heldout and unscored are held out. The test set is separate.")
    json.dump(grpo_split, open(SPLIT_OUT, "w"))
    print(f"[held-out] split saved: {SPLIT_OUT}")

_all = {**{p: i for i, p in enumerate(candidates["pid"])}}
train_dataset = candidates.select([_all[p] for p in grpo_split["train"]])
if train3 is not None:                            # TRAIN_MIX_3: the 3-number training puzzles ([common])
    train_dataset = concatenate_datasets([train_dataset, train3])
trainer.train_dataset = train_dataset           # the dataloader is built in trainer.train(), from this dataset
HELDOUT_STRATA = ["heldout", "unscored"]
CURVE_STRATA = ["heldout", "unscored"]   # the held-out curve of [early stop]

def heldout_dataset(stratum):
    if stratum == "unscored":
        return unscored
    return candidates.select([_all[p] for p in grpo_split[stratum]])

def eval_heldout(trainer, tag_, out_dir):
    """Greedy eval per held-out stratum; one row per stratum."""
    rows = {}
    for s in HELDOUT_STRATA:
        summ, df = eval_puzzles(trainer, heldout_dataset(s), f"heldout_{s}_{tag_}", out_dir=out_dir, quiet=True)
        rows[s] = {k: summ[k] for k in ("accuracy", "accuracy_finished", "truncated", "format_ok", "tokens", "n")}
    return pd.DataFrame(rows).T

print({k: len(grpo_split[k]) for k in HELDOUT_STRATA + ["train"]}, "-> training on", len(train_dataset),
      f"({len(train3)} of them 3-number)" if train3 is not None else "")
_before_dir = f"{GRPO_DIR}/heldout_before_{os.path.basename(SPLIT_OUT)[:-5]}"
if os.path.exists(f"{_before_dir}/summary.csv"):
    heldout_before = pd.read_csv(f"{_before_dir}/summary.csv", index_col=0)
    print(f"[held-out] 'before' cached: {_before_dir}")
else:
    trainer.vllm_generation.sync_weights()      # the starting adapter, not vLLM's base weights
    heldout_before = eval_heldout(trainer, "before", _before_dir)
    heldout_before.to_csv(f"{_before_dir}/summary.csv")
# (per-stratum numbers: heldout_before, saved in _before_dir/summary.csv)

# sampled pass@k on the curve puzzles (CURVE_STRATA), starting adapter: cached next to the greedy "before" eval
from datasets import concatenate_datasets
_per = PASS_N // len(CURVE_STRATA)                   # the first _per puzzles of each curve stratum (a fixed set)
PASS_SET = concatenate_datasets([heldout_dataset(s).select(range(min(_per, len(heldout_dataset(s))))) for s in CURVE_STRATA])
PASS_STRATA = {p: s for s in CURVE_STRATA for p in heldout_dataset(s).select(range(min(_per, len(heldout_dataset(s)))))["pid"]}
if PASS_AT_K:
    _pk = f"{_before_dir}/eval_before_n{len(PASS_SET)}_pass{PASS_AT_K}.csv"
    if os.path.exists(_pk):
        pass_before = pd.read_csv(_pk, index_col=0)
    else:
        trainer.vllm_generation.sync_weights()
        pass_before, _ = pass_at_k(trainer, PASS_SET, f"before_n{len(PASS_SET)}", PASS_AT_K, out_dir=_before_dir,
                                   strata=PASS_STRATA)
else:
    pass_before = None
_mb0 = mixed_before_cached()
report("starting adapter", "starting adapter (cached evals)",
       {"start": {"test4": eval_before, "test3": _mb0["test3"] if _mb0 else None,
                  "curve": curve_of(_before_dir, "before"), "pass": pass_before}})

In [ ]:
# [reward calibration] Inference only. Samples real GRPO groups (CAL_PROMPTS
# training puzzles x num_generations, the starting adapter), scores them with the judge (raw mean and sum) and
# the format rewards, and sets for each judge arm (mean, sum; saved to CALIBRATION_PATH, shared by all arms):
#   floor    1st percentile of real scores (lower scores are clipped to it: caps outliers)
#   missing  two levels, chosen at reward time by MISSING_AT: "below_floor" (5% of the median-to-floor distance below
#            the floor: no solution / over the cap is worse than any scored answer) and "wrong_median" (the median value
#            of the scored wrong answers)
#   coef     TARGET_RATIO x the anchor's within-group spread / the value's within-group spread (with
#            scale_rewards=False a reward's push inside a group is proportional to that spread). Anchor (SPREAD_ANCHOR):
#            "correctness" (default): the correctness reward's spread on the same groups, so the plausibility arms push
#            as hard as the correctness arm; "format": the format rewards' spread. The trained weight is set below
#            (JUDGE_WEIGHT_ANCHOR), from the saved samples
# Also reports the judge's within-puzzle win rate on these groups (as a check against Consistency_Countdown).
# Skipped when CALIBRATION_PATH has every judge arm (RECALIBRATE = True redoes it; the sampling is seeded).
RECALIBRATE = False
_saved_cal = json.load(open(CALIBRATION_PATH)) if os.path.exists(CALIBRATION_PATH) else None
_cal_complete = lambda c: all(k in c and "missing_wrong_median" in c[k] for k in JUDGE_ARMS)
if _saved_cal is not None and not RECALIBRATE and _cal_complete(_saved_cal):
    CAL = _saved_cal
    print(f"[reward calibration] skipped: using {CALIBRATION_PATH} (RECALIBRATE = True to redo)")
else:   # no file, RECALIBRATE, or a file from before an arm existed (seeded: the old entries come out the same)
    if _saved_cal is not None and not RECALIBRATE:
        print(f"[reward calibration] {CALIBRATION_PATH} lacks entries: redoing it")
    CAL_PROMPTS, TARGET_RATIO, FLOOR_Q, SPREAD_ANCHOR = (16 if SMOKE_RUN else 128), 1.0, 0.01, "correctness"
    G = trainer.args.num_generations
    trainer.vllm_generation.sync_weights()
    cal = train_dataset.shuffle(seed=1).select(range(min(CAL_PROMPTS, len(train_dataset))))   # [held-out]'s order, not [train]'s
    ids = [chat_prompt_ids(tokenizer, p) for p in cal["prompt"]]
    outs = trainer.vllm_generation.llm.generate(
        [{"prompt_token_ids": x} for x in ids],
        SamplingParams(n=G, temperature=trainer.args.temperature, max_tokens=MAX_COMPLETION_LENGTH, seed=0), use_tqdm=True)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(outs, ids)), "vLLM returned outputs for other prompts"
    prompts = [p for p in cal["prompt"] for _ in range(G)]
    numbers = [n for n in cal["numbers"] for _ in range(G)]
    targets = [t for t in cal["target"] for _ in range(G)]
    texts = [c.text for o in outs for c in o.outputs]
    completions = [[{"role": "assistant", "content": t}] for t in texts]
    df = pd.DataFrame({"group": np.repeat(np.arange(len(cal)), G), "completion": texts,
                       "correctness": R.correctness_reward_func(prompts, completions, numbers, targets)})
    df["correct"] = df.correctness > 0
    df["format_total"] = sum(np.array(f(completions=completions, numbers=numbers, target=targets), dtype=float)
                             for f in FORMAT_FUNCS)
    df["mean"], df["sum"] = judge_raw(trainer, prompts, completions)
    present = df["mean"].notna()
    wrong_scored = present & ~df.correct
    wrong_median = lambda col, fallback: float(df.loc[wrong_scored, col].median()) if wrong_scored.any() else fallback

    def within_std(col, mask):
        return df[mask].groupby("group")[col].std(ddof=0).mean()

    def win(col, mask):
        rate, pairs = within_puzzle_win_rate(df.group[mask].to_numpy(), df.loc[mask, col].to_numpy(), df.correct[mask].to_numpy())
        return rate, pairs

    _all_rows = np.ones(len(df), dtype=bool)
    spreads = {"correctness": within_std("correctness", _all_rows), "format": within_std("format_total", _all_rows)}
    target_spread = TARGET_RATIO * spreads[SPREAD_ANCHOR]
    assert target_spread > 1e-6, f"the {SPREAD_ANCHOR} reward has no spread within groups: {spreads}"
    m = df.loc[present, "mean"]
    floor_m, med_m = float(m.quantile(FLOOR_Q)), float(m.median())
    df["v_mean"] = df["mean"].clip(lower=floor_m)
    cal_mean = {"floor": floor_m, "missing_below_floor": floor_m - 0.05 * (med_m - floor_m),
                "coef": float(target_spread / within_std("v_mean", present))}
    cal_mean["missing_wrong_median"] = wrong_median("v_mean", cal_mean["missing_below_floor"])

    sm = df.loc[present, "sum"]
    floor_sum, med_sum = float(sm.quantile(FLOOR_Q)), float(sm.median())
    df["v_sum"] = df["sum"].clip(lower=floor_sum)
    cal_sum = {"floor": floor_sum, "missing_below_floor": floor_sum - 0.05 * (med_sum - floor_sum),
               "coef": float(target_spread / within_std("v_sum", present))}
    cal_sum["missing_wrong_median"] = wrong_median("v_sum", cal_sum["missing_below_floor"])

    print(f"{len(df)} completions = {len(cal)} puzzles x {G}; accuracy {df.correct.mean():.3f}; "
          f"scored (solution within {PLAUSIBILITY_MAX_SOLUTION_TOKENS} tokens) {present.mean():.3f}")
    print(f"within-group spread: correctness {spreads['correctness']:.4f}, format {spreads['format']:.4f} "
          f"(format mean {df.format_total.mean():.3f}); anchor {SPREAD_ANCHOR} x {TARGET_RATIO} = {target_spread:.4f}")
    for name, col, c in (("mean", "v_mean", cal_mean), ("sum", "v_sum", cal_sum)):
        print(f"  {name:10s}: within-group spread {within_std(col, present):.4f} -> coef {c['coef']:.4g}; missing "
              f"{c['missing_below_floor']:.4g} (below floor) / {c['missing_wrong_median']:.4g} (wrong median)")
    print("judge win rate (correct over wrong, same puzzle) on these groups: "
          + ", ".join(f"{col} {win(col, present & df[col].notna())[0]:.3f} ({win(col, present & df[col].notna())[1]} pairs)"
                      for col in ("mean", "sum")))
    CAL = {"mean": cal_mean, "sum": cal_sum, "spread_anchor": SPREAD_ANCHOR, "target_ratio": TARGET_RATIO,
           "spreads": spreads, "cal_prompts": len(cal), "accuracy": float(df.correct.mean()),
           "solution_rate": float(present.mean()), "max_solution_tokens": PLAUSIBILITY_MAX_SOLUTION_TOKENS,
           "adapter": ADAPTER_PATH, "judge": JUDGE_ADAPTER_PATH,
           "policy_start": "sft", "created": datetime.now().isoformat(timespec="seconds")}
    os.makedirs(GRPO_DIR, exist_ok=True)
    json.dump(CAL, open(CALIBRATION_PATH, "w"), indent=2)
    df.to_csv(CALIBRATION_PATH.replace(".json", "_samples.csv"), index=False)
    print(f"saved {CALIBRATION_PATH}")

# JUDGE_WEIGHT_ANCHOR "format" ([config]): the trained weight from the saved samples, in memory only (CAL[ARM]["coef"];
# the file's value stays as "coef_saved"). Saved with the run's config.json as JUDGE_WEIGHT.
if ARM in JUDGE_ARMS and JUDGE_WEIGHT_ANCHOR == "format":
    _cs = pd.read_csv(CALIBRATION_PATH.replace(".json", "_samples.csv"))
    _pres = _cs["mean"].notna()
    _sd = lambda d, col: float(np.sqrt(d.groupby("group")[col].var(ddof=0).mean()))
    JUDGE_FORMAT_SD, JUDGE_VALUE_SD = _sd(_cs, "format_total"), _sd(_cs[_pres], f"v_{ARM}")
    CAL[ARM]["coef_saved"] = CAL[ARM].get("coef_saved", CAL[ARM]["coef"])
    CAL[ARM]["coef"] = JUDGE_WEIGHT_SCALE * JUDGE_FORMAT_SD / JUDGE_VALUE_SD
    print(f"[judge weight] format anchor: {JUDGE_WEIGHT_SCALE:g} x format SD {JUDGE_FORMAT_SD:.4f} (all {len(_cs)}) / "
          f"{ARM} SD {JUDGE_VALUE_SD:.3f} (scored {int(_pres.sum())}) = {CAL[ARM]['coef']:.4g} "
          f"(calibration file's correctness-anchored coef {CAL[ARM]['coef_saved']:.4g})")
JUDGE_WEIGHT = plausibility_weight()

# apply to the current trainer (the plausibility reward reads CAL at call time; the weight is set here)
for i, f in enumerate(trainer.reward_funcs):
    if getattr(f, "__name__", "") == "plausibility":
        trainer.reward_weights[i] = plausibility_weight()
print(f"ARM={ARM}: plausibility weight {plausibility_weight():.4g}; "
      + (f"missing = {MISSING_AT} ({CAL[ARM]['missing_' + MISSING_AT]:.4g}); " if ARM in JUDGE_ARMS else "")
      + ("trained" if ARM in JUDGE_ARMS else "logged only (weight 0)"))

In [ ]:
# [early stop] Every checkpoint is kept, so the best
# can be chosen afterwards by the held-out curve.
#   held-out curve  every HELDOUT_EVAL_STEPS steps: greedy accuracy on CURVE_STRATA (the held-out set from the
#                   candidates + unscored; step 0 = the cached
#                   [held-out] "before" eval). Stop when it dropped in two consecutive evals AND the latest is STOP_DROP
#                   below the best (single dips are noise). Labels are used only for this model selection, never in the
#                   reward (except the correctness arm); same rule for all arms.
#   collapse guards label-free, from the training logs (every 5 steps): more than TRUNC_STOP of completions truncated,
#                   or (judge arms) more than ZERO_STD_STOP of groups with zero reward variance,
#                   for GUARD_LOGS logs
#   pass@k          at PASS_EVAL_STEPS: PASS_AT_K samples on the PASS_N puzzles of [held-out] (LOG_DIR/pass_curve.csv)
#   rich evals      at RICH_EVAL_STEPS: the end-of-run evals (test set, held-out strata, mixed 3/4-number test), tagged
#                   step{N} in LOG_DIR (skipped when eval_test_step{N}.json exists: survives a resume)
# The curve is saved to LOG_DIR/heldout_curve.csv and survives a resume.
from datasets import concatenate_datasets

HELDOUT_EVAL_STEPS = 5 if SMOKE_RUN else 25
STOP_DROP = 1.0                  # off for the long validation run (all checkpoints are kept)
TRUNC_STOP, ZERO_STD_STOP, GUARD_LOGS = 0.5, 0.5, 2
CURVE_SET = concatenate_datasets([heldout_dataset(s) for s in CURVE_STRATA])

def curve_eval(trainer):
    trainer.vllm_generation.sync_weights()
    s, _ = eval_puzzles(trainer, CURVE_SET, "curve", quiet=True)
    return {k: float(s[k]) for k in ("accuracy", "accuracy_finished", "truncated", "format_ok", "tokens")}

def start_results():
    """The starting adapter's results for the report's "start" column (all cached)."""
    mb = mixed_before_cached()
    return {"test4": eval_before, "test3": mb["test3"] if mb else None,
            "curve": {**early_stop.curve[0], "n": len(CURVE_SET)}, "pass": pass_before}

_CURVE_KEYS = ("accuracy", "accuracy_finished", "truncated", "format_ok", "tokens")

class EarlyStop(TrainerCallback):
    def __init__(self, trainer, path):
        self.trainer, self.path, self.flags = trainer, path, 0
        if os.path.exists(path):
            self.curve = pd.read_csv(path).to_dict("records")
        else:   # step 0 from the cached "before" held-out eval (same greedy settings, same puzzles)
            b = pd.concat([pd.read_csv(f"{_before_dir}/eval_heldout_{s}_before_samples.csv") for s in CURVE_STRATA
                           if os.path.exists(f"{_before_dir}/eval_heldout_{s}_before_samples.csv")])
            self.curve = [{"step": 0, "accuracy": float(b.correct.mean()),
                           "accuracy_finished": float(b.loc[~b.truncated.astype(bool), "correct"].mean()),
                           "truncated": float(b.truncated.mean()), "format_ok": float(b.format_ok.mean()),
                           "tokens": float(b.tokens.mean())}]

    def _stop(self, control, why):
        print(f"[early stop] step {self.trainer.state.global_step}: {why}")
        control.should_training_stop = True
        control.should_save = True

    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        here = {}                                   # this step's results, printed as ONE report below
        _pass_path = f"{LOG_DIR}/pass_curve.csv"
        if PASS_AT_K and step in PASS_EVAL_STEPS and not (os.path.exists(_pass_path)
                                                          and step in set(pd.read_csv(_pass_path).step)):
            self.trainer.vllm_generation.sync_weights()
            s, _ = pass_at_k(self.trainer, PASS_SET, f"step{step}_n{len(PASS_SET)}", PASS_AT_K, out_dir=LOG_DIR,
                             strata=PASS_STRATA)
            new = s.reset_index().rename(columns={"index": "set"}).assign(step=step)
            rows = pd.concat(([pd.read_csv(_pass_path)] if os.path.exists(_pass_path) else []) + [new])
            rows.to_csv(_pass_path, index=False)
            here["pass"] = s
        if step in RICH_EVAL_STEPS and not os.path.exists(f"{LOG_DIR}/eval_test_step{step}.json"):
            self.trainer.vllm_generation.sync_weights()
            s, d = eval_puzzles(self.trainer, _test, f"test_step{step}", out_dir=LOG_DIR, quiet=True)
            h = eval_heldout(self.trainer, f"step{step}", LOG_DIR)
            h.to_csv(f"{LOG_DIR}/heldout_step{step}_summary.csv")
            _, s3, _ = eval_mixed(self.trainer, f"step{step}", out_dir=LOG_DIR, test4=(s, d))   # 4-number eval reused
            here.update(test4=s, test3=s3, curve=curve_of(LOG_DIR, f"step{step}"))
        new_point = not (step % HELDOUT_EVAL_STEPS or any(r["step"] == step for r in self.curve))
        if new_point:   # the curve point: from the rich eval's held-out eval when there was one (same puzzles, greedy)
            c = here.get("curve") or curve_eval(self.trainer)
            self.curve.append({"step": step, **{k: c[k] for k in _CURVE_KEYS}})
            pd.DataFrame(self.curve).to_csv(self.path, index=False)
        if here:
            report(f"step {step}", f"training weights at step {step}", {"start": start_results(), f"step {step}": here},
                   out=f"{LOG_DIR}/report_step{step}.csv")
        if not new_point:
            return
        accs = [r["accuracy"] for r in self.curve]
        print(curve_line(self.curve))
        if len(accs) >= 3 and accs[-1] < accs[-2] < accs[-3] and accs[-1] <= max(accs) - STOP_DROP:
            self._stop(control, f"held-out accuracy fell twice in a row and is {max(accs) - accs[-1]:.3f} below the best")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "reward" not in logs:
            return
        trunc, zero = logs.get("completions/clipped_ratio", 0.0), logs.get("frac_reward_zero_std", 0.0)
        # zero reward variance is collapse only when a continuous reward is trained (saturated format rewards give it
        # by design in the control arm)
        zero_bad = zero > ZERO_STD_STOP and ARM in JUDGE_ARMS
        self.flags = self.flags + 1 if (trunc > TRUNC_STOP or zero_bad) else 0
        if self.flags >= GUARD_LOGS:
            self._stop(control, f"collapse guard: truncated {trunc:.2f}, zero-variance groups {zero:.2f} "
                                f"({GUARD_LOGS} logs in a row)")

trainer.callback_handler.callbacks = [c for c in trainer.callback_handler.callbacks if type(c).__name__ != "EarlyStop"]
early_stop = EarlyStop(trainer, f"{LOG_DIR}/heldout_curve.csv")
trainer.add_callback(early_stop)
print(f"early stop installed: held-out curve every {HELDOUT_EVAL_STEPS} steps on {len(CURVE_SET)} puzzles "
      f"(step 0 accuracy {early_stop.curve[0]['accuracy']:.3f}); guards truncated > {TRUNC_STOP}, "
      f"zero-variance groups > {ZERO_STD_STOP} for {GUARD_LOGS} logs")

In [ ]:
# [train] Resumes from the newest checkpoint in LOG_DIR if there is one (same ARM x RUN_TAG).
import glob, time
if ARM in JUDGE_ARMS:
    _w = [float(w) for f, w in zip(trainer.reward_funcs, trainer.reward_weights) if getattr(f, "__name__", "") == "plausibility"]
    assert CAL is not None and _w and _w[0] > 0, "plausibility arm without a calibration: run [reward calibration] first"
assert "grpo_split" in globals() and (trainer.train_dataset is train_dataset or trainer.train_dataset is globals().get("train_ordered")), \
    "run [held-out] first"
if INTERLEAVE_34:   # per step: half four-number, then half three-number puzzles, each size in its own seeded order (SEED / SEED + 1)
    assert not trainer.args.shuffle_dataset, "INTERLEAVE_34 needs shuffle_dataset False: rerun [build] and the cells after it"
    _per = COMPLETIONS_PER_STEP // NUM_GENERATIONS
    _by = {n: np.random.default_rng(SEED + (n == 3)).permutation([i for i, x in enumerate(train_dataset["numbers"]) if len(x) == n])
           for n in (4, 3)}
    assert len(_by[4]) == len(_by[3]) == len(train_dataset) // 2, {n: len(v) for n, v in _by.items()}
    _order = [int(v[b * _per // 2 + j]) for b in range(len(_by[4]) * 2 // _per) for v in (_by[4], _by[3]) for j in range(_per // 2)]
    train_ordered = train_dataset.select(_order)
    assert all(len(x) == (4 if i % _per < _per // 2 else 3) for i, x in enumerate(list(train_ordered["numbers"])[:MAX_STEPS * _per]))
    trainer.train_dataset = train_ordered
    os.makedirs(LOG_DIR, exist_ok=True)
    pd.DataFrame({"step": [i // _per + 1 for i in range(len(train_ordered))], "pid": list(train_ordered["pid"]),
                  "n_numbers": [len(x) for x in train_ordered["numbers"]]}).to_csv(f"{LOG_DIR}/train_order.csv", index=False)
    print(f"[train] INTERLEAVE_34: {_per // 2} four- then {_per // 2} three-number puzzles per step "
          f"({len(train_ordered)} puzzles in order) -> {LOG_DIR}/train_order.csv")
assert not set(trainer.train_dataset["pid"]) & set(grpo_split[CURVE_STRATA[0]]), "held-out puzzles in training"
assert any(type(c).__name__ == "EarlyStop" for c in trainer.callback_handler.callbacks), "run [early stop] first"

# checker labels this run consumes for training (evaluation not counted)
_labels = {}
if ARM == "correctness":
    _labels["checker_training_completions"] = MAX_STEPS * COMPLETIONS_PER_STEP
if ARM in JUDGE_ARMS:
    _jl = f"{os.path.dirname(JUDGE_ADAPTER_PATH)}/labels.json"
    _labels["judge"] = json.load(open(_jl)) if os.path.exists(_jl) else "unknown"
    _labels["checker_calibration_completions"] = (16 if SMOKE_RUN else 128) * NUM_GENERATIONS
os.makedirs(LOG_DIR, exist_ok=True)
json.dump(_labels, open(f"{LOG_DIR}/labels.json", "w"), indent=1)
print("checker labels for training:", _labels)

_ckpts = sorted(glob.glob(f"{LOG_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
resume = _ckpts[-1] if _ckpts else None

# Resume check: the trainer's own resume restored step, optimizer and scheduler but NOT the policy adapter in the
# 2026-10-21 continuation (mean-34-cont100: its checkpoint-125 was 0.38 from the SFT adapter, 0.70 from checkpoint-100;
# KL 0.0006 at step 105). Cause not found. So at the start of training (after the trainer's resume) the policy adapter
# is compared with the checkpoint's; if they differ, the checkpoint's weights are loaded (in place: the optimizer keeps
# its parameters), checked again, and synced to vLLM. Training stops if they still differ.
from peft import get_peft_model_state_dict, set_peft_model_state_dict
from safetensors.torch import load_file

class ResumeCheck(TrainerCallback):
    def __init__(self, trainer, ckpt):
        self.trainer, self.ckpt = trainer, ckpt

    def _max_diff(self, want):
        now = get_peft_model_state_dict(self.trainer.model, adapter_name="default")
        assert set(now) == set(want), (f"policy adapter keys differ from {self.ckpt}: "
                                       f"{sorted(set(now) ^ set(want))[:4]}")
        return max((now[k].detach().float().cpu() - want[k].to(now[k].dtype).float()).abs().max().item() for k in want)

    def on_train_begin(self, args, state, control, **kwargs):
        want = load_file(f"{self.ckpt}/adapter_model.safetensors")
        before = self._max_diff(want)
        if before > 0:
            res = set_peft_model_state_dict(self.trainer.model, want, adapter_name="default")
            assert not res.unexpected_keys, res.unexpected_keys[:4]
            after = self._max_diff(want)
            assert after == 0, f"[resume check] the policy adapter still differs from {self.ckpt} (max |diff| {after:g})"
            self.trainer.vllm_generation.sync_weights()
        assert all(p.requires_grad for n, p in self.trainer.model.named_parameters() if ".default." in n and "lora_" in n), \
            "policy adapter not trainable after the resume check"
        print(f"[resume check] step {state.global_step}: policy adapter vs {os.path.basename(self.ckpt)}: max |diff| "
              f"{before:g}" + (" -> loaded it (the trainer's resume had not), synced vLLM" if before > 0 else " (matches)"))
        self.first_step = state.global_step + 1

    def on_step_end(self, args, state, control, **kwargs):
        # after the first step: one update away from the checkpoint, far from the starting adapter (L2 over the adapter)
        if state.global_step != getattr(self, "first_step", None):
            return
        now = get_peft_model_state_dict(self.trainer.model, adapter_name="default")
        dist = lambda w: sum(((now[k].detach().float().cpu() - w[k].float()) ** 2).sum().item() for k in w) ** 0.5
        to_ckpt = dist(load_file(f"{self.ckpt}/adapter_model.safetensors"))
        to_start = dist(load_file(f"{_policy_init}/adapter_model.safetensors"))
        print(f"[resume check] step {state.global_step}: L2 to {os.path.basename(self.ckpt)} {to_ckpt:.3f}, "
              f"to the starting adapter {to_start:.3f}")
        if to_ckpt >= to_start:
            control.should_training_stop = True
            print("[resume check] STOP: the policy is nearer the starting adapter than the checkpoint it resumed from")

trainer.callback_handler.callbacks = [c for c in trainer.callback_handler.callbacks if type(c).__name__ != "ResumeCheck"]
if resume:
    trainer.add_callback(ResumeCheck(trainer, resume))
print(f"{'resuming from ' + resume if resume else 'starting fresh'}; {len(trainer.train_dataset)} training puzzles; "
      f"weights {dict(zip([getattr(f, '__name__', '?') for f in trainer.reward_funcs], trainer.reward_weights.tolist()))}")
_t0, _s0 = time.time(), (int(resume.rsplit('-', 1)[1]) if resume else 0)
# the [config] values this launch used (the run name doesn't carry all of them, e.g. SCALE_REWARDS); a resume
# writes its own file next to the first
os.makedirs(LOG_DIR, exist_ok=True)
json.dump({**{k: v for k, v in globals().items() if k.isupper() and not k.startswith('_')
              and isinstance(v, (bool, int, float, str, tuple, list, type(None)))},
           "trainer.scale_rewards": str(trainer.args.scale_rewards), "resumed_from": resume},
          open(f"{LOG_DIR}/config{f'_resume{_s0}' if resume else ''}.json", "w"), indent=1, default=str)

print(CALIBRATION_PATH.rsplit("/", 1)[1], "plausibility weight", [w for f, w in zip(trainer.reward_funcs, trainer.reward_weights) if f.__name__ == "plausibility"])


trainer.train(resume_from_checkpoint=resume)

_steps = trainer.state.global_step - _s0
print(f"{_steps} steps in {(time.time() - _t0) / 60:.1f} min = {(time.time() - _t0) / max(_steps, 1):.0f} s/step")

trainer.vllm_generation.sync_weights()   # vLLM was last synced before the final optimizer step
CURRENT_WEIGHTS = f"final weights of {os.path.basename(LOG_DIR)} (step {trainer.state.global_step})"
# all saved as before (eval_test_after, eval_test3/4_after, eval_heldout_*_after, eval_after_n*_pass*); one report printed
eval_after, _after_df = eval_puzzles(trainer, _test, "test_after", out_dir=LOG_DIR, quiet=True)
_, test3_after, _ = eval_mixed(trainer, "after", LOG_DIR, test4=(eval_after, _after_df))   # the 4-number eval reused
heldout_after = eval_heldout(trainer, "after", LOG_DIR)      # per stratum (saved); the report shows the curve set
pass_after = (pass_at_k(trainer, PASS_SET, f"after_n{len(PASS_SET)}", PASS_AT_K, out_dir=LOG_DIR, strata=PASS_STRATA)[0]
              if PASS_AT_K else None)
mixed_before_evals(trainer)                                  # cached in GRPO_DIR (computed once if missing)
report("after training", CURRENT_WEIGHTS,
       {"start": start_results(), "final": {"test4": eval_after, "test3": test3_after,
                                            "curve": curve_of(LOG_DIR, "after"), "pass": pass_after}},
       out=f"{LOG_DIR}/report_after.csv")

curve = pd.DataFrame(early_stop.curve)
best = curve.loc[curve["accuracy"].idxmax()]
with pd.option_context("display.float_format", "{:.3f}".format):
    print(f"\n[held-out curve] held-out curve puzzles ({' + '.join(CURVE_STRATA)}), greedy, {len(CURVE_SET)} x 1, "
          f"training weights by step:\n", curve.to_string(index=False))
print(f"best on the held-out curve: step {int(best.step)} (accuracy {best.accuracy:.3f})"
      + (f" -> {LOG_DIR}/checkpoint-{int(best.step)}" if best.step > 0 else " = the starting adapter"))

In [ ]:
# [checkpoint eval] (optional, inference only) CKPT: a saved step of this run (LOG_DIR/checkpoint-CKPT), or None to skip.
# Loads it, evals the 4- and 3-number test sets and the held-out curve puzzles (greedy) and pass@k (sampled), prints ONE
# report next to the starting adapter, then RESTORES the session's weights, so later cells still measure CURRENT_WEIGHTS
# (before 2026-11-09 this cell left the checkpoint loaded and [mixed test] then measured it). Saved in
# LOG_DIR/eval_checkpoint-CKPT (test_ckpt, test3/4_ckpt, heldout_*_ckpt, ckptCKPT_n*_pass*, report.csv).
import safetensors.torch
from peft import get_peft_model_state_dict, set_peft_model_state_dict

CKPT = 75

if CKPT is None:
    print("[checkpoint eval] skipped (CKPT = None)")
else:
    ckpt_dir, out_dir = f"{LOG_DIR}/checkpoint-{CKPT}", f"{LOG_DIR}/eval_checkpoint-{CKPT}"
    os.makedirs(out_dir, exist_ok=True)
    _now = {k: v.detach().clone() for k, v in get_peft_model_state_dict(trainer.model, adapter_name="default").items()}
    res = set_peft_model_state_dict(trainer.model, safetensors.torch.load_file(f"{ckpt_dir}/adapter_model.safetensors"),
                                    adapter_name="default")
    assert not res.unexpected_keys, res.unexpected_keys[:4]
    trainer.vllm_generation.sync_weights()
    try:
        _ct, _cd = eval_puzzles(trainer, _test, "test_ckpt", out_dir=out_dir, quiet=True)
        _, _c3, _ = eval_mixed(trainer, "ckpt", out_dir, test4=(_ct, _cd))
        eval_heldout(trainer, "ckpt", out_dir)
        _cp = (pass_at_k(trainer, PASS_SET, f"ckpt{CKPT}_n{len(PASS_SET)}", PASS_AT_K, out_dir=out_dir,
                         strata=PASS_STRATA)[0] if PASS_AT_K else None)
    finally:
        set_peft_model_state_dict(trainer.model, _now, adapter_name="default")
        trainer.vllm_generation.sync_weights()
    report(f"checkpoint {CKPT}", f"{os.path.basename(LOG_DIR)}/checkpoint-{CKPT} (session weights restored after: "
           f"{CURRENT_WEIGHTS})",
           {"start": start_results(), f"checkpoint {CKPT}": {"test4": _ct, "test3": _c3, "curve": curve_of(out_dir, "ckpt"),
                                                            "pass": _cp}}, out=f"{out_dir}/report.csv")


In [ ]:
# [mixed test] (optional, inference only) [train] already reports Gandhi et al. 2025's Countdown mix for the final weights
# (3-number test3_set + the 4-number test set, greedy; their Llama-3.2-3B ~30% after PPO, ~60% primed). This cell evals
# the session's weights (CURRENT_WEIGHTS, named in the header) and saves in LOG_DIR/eval_mixed_session (not *_after).
trainer.vllm_generation.sync_weights()
_mb = mixed_before_evals(trainer)
_out = f"{LOG_DIR}/eval_mixed_session"
os.makedirs(_out, exist_ok=True)
mixed_session, _s3, _s4 = eval_mixed(trainer, "session", _out)
report("mixed test", CURRENT_WEIGHTS, {"start": {"test4": _mb["test4"], "test3": _mb["test3"]},
                                       "session": {"test4": _s4, "test3": _s3}}, out=f"{_out}/report.csv")


In [ ]:
# [5-number test] (optional, inference only) Greedy accuracy on N_TEST5 generated 5-number puzzles (test5_set, [common]):
# a size never trained on, by SFT, the judge or GRPO. Does training on 3- and 4-number puzzles carry over? "before" is the
# starting adapter (loaded for the eval, then the current weights are restored; cached in GRPO_DIR), "after" the weights
# in the session.
from peft import get_peft_model_state_dict, set_peft_model_state_dict
from safetensors.torch import load_file

_t5_cols = ["accuracy", "truncated", "format_ok", "tokens"]
_t5_before = f"{GRPO_DIR}/eval_test5_before_n{len(test5_set)}_max{MAX_COMPLETION_LENGTH}"
if os.path.exists(f"{_t5_before}.json"):
    test5_before = json.load(open(f"{_t5_before}.json"))
    _t5_before_df = pd.read_csv(f"{_t5_before}_samples.csv")
else:
    _now = {k: v.detach().clone() for k, v in get_peft_model_state_dict(trainer.model, adapter_name="default").items()}
    set_peft_model_state_dict(trainer.model, load_file(f"{_policy_init}/adapter_model.safetensors"), adapter_name="default")
    trainer.vllm_generation.sync_weights()
    try:
        test5_before, _t5_before_df = eval_puzzles(trainer, test5_set, "test5_before", out_dir=GRPO_DIR)
    finally:
        set_peft_model_state_dict(trainer.model, _now, adapter_name="default")
        trainer.vllm_generation.sync_weights()
    for ext in (".json", "_samples.csv"):
        shutil.move(f"{GRPO_DIR}/eval_test5_before{ext}", f"{_t5_before}{ext}")

trainer.vllm_generation.sync_weights()
test5_after, _t5_after_df = eval_puzzles(trainer, test5_set, "test5_after", out_dir=LOG_DIR)
_gained = int((~_t5_before_df.correct.astype(bool).values & _t5_after_df.correct.values).sum())
_lost = int((_t5_before_df.correct.astype(bool).values & ~_t5_after_df.correct.values).sum())
with pd.option_context("display.float_format", "{:.3f}".format):
    print(f"\n[5-number test] generated 5-number puzzles, greedy, {len(test5_set)} x 1 | weights: start = starting adapter, "
          f"after = {CURRENT_WEIGHTS}; solved by after only {_gained}, by before only {_lost}:")
    print(pd.DataFrame({"before": {k: test5_before[k] for k in _t5_cols}, "after": {k: test5_after[k] for k in _t5_cols}}))
    print("\nwhat is wrong (share of the 5-number puzzles):")
    print(pd.DataFrame({"before": _t5_before_df.kind.value_counts(normalize=True),
                        "after": _t5_after_df.kind.value_counts(normalize=True)}).fillna(0))